# 🧠 Deep Learning untuk Analisis Sentimen Kesehatan Mental

**Workshop Pemrosesan Bahasa Alami (PBA)**  
_Institut Teknologi Sumatera (ITERA)_

---

## Tujuan Workshop

Pada sesi ini kita akan membangun dan membandingkan **tiga model deep learning** untuk mengklasifikasikan status kesehatan mental seseorang berdasarkan teks:

| # | Model | Konsep Utama |
|---|-------|--------------|
| 1 | **BiLSTM** | Recurrent network dua arah |
| 2 | **BiLSTM + Attention** | Mekanisme perhatian (attention) pada kata-kata penting |
| 3 | **DistilBERT fine-tuning** | Transfer learning dari model pre-trained Transformer |

**Dataset:** [Sentiment Analysis for Mental Health](https://www.kaggle.com/datasets/suchintikasarkar/sentiment-analysis-for-mental-health)  
**7 Kelas:** Normal · Depression · Suicidal · Anxiety · Stress · Bipolar · Personality Disorder  
**Framework:** PyTorch + HuggingFace Transformers

---

## Arsitektur yang Dikomparasi

```
Teks Input
    │
    ├─▶  [BiLSTM]  ──────────────────────────────────────────▶ Klasifikasi
    │    Embedding → BiLSTM(2 layer) → Last Hidden → FC
    │
    ├─▶  [BiLSTM + Attention]  ──────────────────────────────▶ Klasifikasi
    │    Embedding → BiLSTM(2 layer) → Attention → Weighted Sum → FC
    │
    └─▶  [DistilBERT Fine-tuning]  ──────────────────────────▶ Klasifikasi
         Tokenizer → DistilBERT (pre-trained) → [CLS] Repr. → FC
```

---

> 💡 **Catatan:** Klik ▶ atau tekan `Shift+Enter` untuk menjalankan setiap cell.

## 📦 Bagian 1: Instalasi & Persiapan

Kita menggunakan **`uv`** sebagai package manager. Jalankan cell berikut untuk memastikan semua dependensi terinstal.

In [ ]:
# Instal dependensi (jalankan sekali)
import subprocess, sys

packages = [
    "torch", "transformers", "kagglehub", "opendatasets",
    "pandas", "numpy", "scikit-learn",
    "matplotlib", "seaborn", "ipywidgets",
]

for pkg in packages:
    try:
        __import__(pkg.replace("-", "_").split("[")[0])
        print(f"  ✅ {pkg} sudah terinstal")
    except ImportError:
        print(f"  📥 Menginstal {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("\n🎉 Semua paket siap!")

In [ ]:
# Import semua modul yang diperlukan
import os
import sys
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("inline")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn

# Pastikan kita berada di direktori module_DL
module_dir = os.path.dirname(os.path.abspath(".")) if os.path.basename(os.getcwd()) != "module_DL" else os.getcwd()
# Coba cari direktori module_DL
for candidate in [".", os.path.dirname(os.getcwd())]:
    if os.path.exists(os.path.join(candidate, "config.py")):
        sys.path.insert(0, os.path.abspath(candidate))
        os.chdir(os.path.abspath(candidate))
        break

from config import DEVICE, LABEL_LIST, NUM_CLASSES, RANDOM_SEED
from train import set_seed

# Set seed untuk reproducibility
set_seed(RANDOM_SEED)

print(f"🖥️  Device    : {DEVICE}")
print(f"🌱 Seed       : {RANDOM_SEED}")
print(f"📂 Working dir: {os.getcwd()}")
print(f"🏷️  Kelas      : {LABEL_LIST}")
print(f"🔢 Jumlah kelas: {NUM_CLASSES}")

---

## 📊 Bagian 2: Dataset & Eksplorasi Data

### Tentang Dataset

Dataset **"Sentiment Analysis for Mental Health"** merupakan kompilasi dari beberapa sumber (Reddit, Twitter, forum kesehatan) yang sudah diberi label oleh para ahli.

| Fitur | Keterangan |
|-------|------------|
| `statement` | Teks/pernyataan seseorang |
| `status` | Label: salah satu dari 7 status kesehatan mental |

**7 Kelas Target:**
- `Normal` — kondisi mental sehat
- `Depression` — depresi
- `Suicidal` — pikiran bunuh diri
- `Anxiety` — kecemasan
- `Stress` — stres
- `Bipolar` — gangguan bipolar
- `Personality Disorder` — gangguan kepribadian

In [ ]:
# Download dataset dari Kaggle (otomatis via kagglehub)
from download_data import download_dataset

csv_path = download_dataset()
print(f"\n📄 Dataset tersimpan di: {csv_path}")

### Eksplorasi Data (EDA)

Sebelum melatih model, penting untuk memahami:
1. Distribusi kelas (seimbang atau tidak?)
2. Panjang teks (mempengaruhi `MAX_LEN`)
3. Contoh teks per kelas

In [ ]:
# Baca raw data untuk eksplorasi
df_raw = pd.read_csv(csv_path)
df_raw.columns = df_raw.columns.str.strip()

print(f"Shape dataset : {df_raw.shape}")
print(f"\nKolom        : {df_raw.columns.tolist()}")
print(f"\nMissing values:")
print(df_raw.isnull().sum())
print(f"\nContoh data:")
df_raw.head(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Distribusi kelas ──
dist = df_raw["status"].value_counts()
colors = sns.color_palette("Set2", len(dist))
axes[0].barh(dist.index, dist.values, color=colors)
axes[0].set_title("Distribusi Kelas", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Jumlah Data")
for i, (v, lbl) in enumerate(zip(dist.values, dist.index)):
    axes[0].text(v + 100, i, str(v), va="center")

# ── Distribusi panjang teks ──
text_lens = df_raw["statement"].dropna().astype(str).apply(lambda x: len(x.split()))
axes[1].hist(text_lens.clip(upper=200), bins=50, color="steelblue", edgecolor="white")
axes[1].axvline(128, color="red", linestyle="--", label="MAX_LEN = 128")
axes[1].set_title("Distribusi Panjang Teks (kata)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Jumlah Kata")
axes[1].set_ylabel("Frekuensi")
axes[1].legend()

plt.suptitle("Eksplorasi Data Mental Health", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print(f"\nStatistik panjang teks (kata):")
print(text_lens.describe().round(1))

In [ ]:
# Contoh satu teks per kelas
from config import LABEL_LIST

print("CONTOH TEKS PER KELAS")
print("=" * 70)
for label in LABEL_LIST:
    subset = df_raw[df_raw["status"].str.strip().str.lower() == label.lower()]
    if len(subset) > 0:
        contoh = subset["statement"].iloc[0]
        print(f"\n[{label}]")
        print(f"  {str(contoh)[:200]}..." if len(str(contoh)) > 200 else f"  {contoh}")

---

## 🧹 Bagian 3: Preprocessing Teks

### Pipeline Pembersihan

Teks mentah perlu dibersihkan sebelum dimasukkan ke model. Pipeline yang digunakan:

1. **Lowercase** — semua huruf kecil
2. **Hapus URL** — link tidak relevan untuk analisis sentimen
3. **Hapus tag HTML** — teks dari web sering mengandung `<br>`, `<p>`, dll.
4. **Hapus karakter non-alfabet** — angka, tanda baca, simbol
5. **Hapus spasi berlebih** — normalisasi whitespace

> ⚠️ Untuk model BiLSTM kita menggunakan teks yang sudah dibersihkan (`cleaned_text`).  
> Untuk DistilBERT, tokenizer BERT menangani banyak hal secara internal, jadi kita gunakan teks asli (`text`).

In [ ]:
from preprocess import load_and_clean, show_cleaning_examples
from config import SAMPLE_SIZE

# Load dan bersihkan data (dengan sampling 15.000 baris)
df = load_and_clean(csv_path, sample_size=SAMPLE_SIZE)

print(f"\nShape setelah preprocessing: {df.shape}")
df[["text", "cleaned_text", "label", "label_encoded"]].head(3)

In [ ]:
# Tampilkan contoh sebelum vs sesudah pembersihan
show_cleaning_examples(df, n=4)

---

## 🔡 Bagian 4: Vocabulary & DataLoader

### Tokenisasi & Vocabulary (untuk BiLSTM)

Model BiLSTM memerlukan konversi teks → urutan angka. Caranya:

1. **Bangun vocabulary** — daftar semua kata unik beserta indeksnya.
2. **Tokenisasi** — setiap kata diganti dengan indeks bilangan bulat.
3. **Padding** — semua urutan disamakan panjangnya ke `MAX_LEN`.

```
Teks:     "i feel very anxious today"
Tokens:   ["i", "feel", "very", "anxious", "today"]
Indices:  [15,  243,   89,    1024,       365,   0, 0, ..., 0]
                                                    ↑ padding
```

Token khusus: `<PAD>` (index 0), `<UNK>` (index 1)

DistilBERT menggunakan tokenizer sendiri (WordPiece) dan tidak memerlukan langkah ini.

In [ ]:
from dataset import Vocabulary, get_lstm_dataloaders
from config import VOCAB_SIZE, MAX_LEN, LSTM_BATCH_SIZE, VOCAB_PATH

# Bangun vocabulary dari teks training
vocab = Vocabulary()
vocab.build_vocab(df["cleaned_text"].tolist(), max_size=VOCAB_SIZE)
vocab.save(VOCAB_PATH)

# Uji konversi teks ke indeks
contoh = "i feel very anxious and stressed all the time"
indices = vocab.text_to_indices(contoh, MAX_LEN)
print(f"\nContoh teks  : '{contoh}'")
print(f"Indices (15 pertama): {indices[:15]}")
print(f"Panjang sequence: {len(indices)}")  # harus = MAX_LEN

In [ ]:
# Buat DataLoaders untuk BiLSTM
train_loader, val_loader, test_loader = get_lstm_dataloaders(
    df, vocab, max_len=MAX_LEN, batch_size=LSTM_BATCH_SIZE
)

# Cek satu batch
batch = next(iter(train_loader))
x, labels, lengths = batch
print(f"\nShape satu batch:")
print(f"  x (token indices): {x.shape}   → (batch_size, seq_len)")
print(f"  labels           : {labels.shape} → (batch_size,)")
print(f"  lengths          : {lengths.shape} → (batch_size,)")
print(f"\nContoh label (indeks): {labels[:5].tolist()}")
print(f"Contoh label (nama) : {[LABEL_LIST[i] for i in labels[:5].tolist()]}")

---

## 🔁 Bagian 5: Model 1 — BiLSTM

### Arsitektur BiLSTM

**LSTM (Long Short-Term Memory)** adalah jenis RNN yang mampu "mengingat" konteks dari urutan panjang.

**Bidirectional LSTM (BiLSTM)** memproses teks dari dua arah:
- **Forward** → membaca kiri ke kanan  
- **Backward** → membaca kanan ke kiri

Dengan dua arah, model dapat menangkap konteks sebelum **dan** sesudah setiap kata.

```
"i feel very anxious"

Forward:   i → feel → very → anxious
Backward:  anxious → very → feel → i

Output: concat(h_forward_last, h_backward_last)
              ↓
         Fully Connected → 7 kelas
```

**Arsitektur lengkap:**
```
Embedding(20000 vocab, 128 dim)
    → Dropout(0.3)
    → BiLSTM(128→256, 2 layer)
    → Concat last hidden (512 dim)
    → Dropout(0.3)
    → Linear(512 → 7)
```

In [ ]:
from models import BiLSTMClassifier, count_parameters
from config import DEVICE

# Inisialisasi model
bilstm = BiLSTMClassifier(vocab_size=len(vocab))
bilstm = bilstm.to(DEVICE)

print("Arsitektur BiLSTM:")
print(bilstm)
print(f"\n📊 Total parameter yang dilatih: {count_parameters(bilstm):,}")

In [ ]:
from train import train_model, get_criterion
from config import LSTM_EPOCHS, LSTM_LR, LSTM_PATIENCE, BILSTM_MODEL_PATH

# Hitung class weights untuk menangani ketidakseimbangan kelas
label_counts = df["label"].value_counts().to_dict()

print("🚀 Mulai training BiLSTM...")
hist_bilstm = train_model(
    model=bilstm,
    train_loader=train_loader,
    val_loader=val_loader,
    model_type="lstm",
    save_path=BILSTM_MODEL_PATH,
    epochs=LSTM_EPOCHS,
    lr=LSTM_LR,
    patience=LSTM_PATIENCE,
    device=DEVICE,
    label_counts=label_counts,
)

In [ ]:
from train import evaluate_lstm, plot_confusion_matrix, print_classification_report

criterion = get_criterion(label_counts)

# Evaluasi pada test set
_, _, preds_bilstm, labels_bilstm = evaluate_lstm(bilstm, test_loader, criterion, DEVICE)

# Confusion matrix
plot_confusion_matrix(labels_bilstm, preds_bilstm, "BiLSTM")

# Classification report
metrics_bilstm = print_classification_report(labels_bilstm, preds_bilstm, "BiLSTM")
metrics_bilstm["training_time_min"] = hist_bilstm["total_time"] / 60

In [ ]:
from train import plot_training_curves

# Plot training curves
plot_training_curves(hist_bilstm, "BiLSTM")

---

## 🎯 Bagian 6: Model 2 — BiLSTM + Attention

### Mekanisme Attention

Model BiLSTM standar hanya menggunakan **hidden state terakhir** sebagai representasi seluruh teks. Ini kurang optimal karena:
- Teks panjang: hidden state terakhir mungkin sudah "lupa" info di awal
- Tidak semua kata sama pentingnya

**Attention** memungkinkan model "memperhatikan" kata-kata berbeda dengan bobot yang berbeda:

$$\alpha_t = \text{softmax}(\tanh(W \cdot h_t))$$

$$c = \sum_t \alpha_t \cdot h_t$$

Di mana:
- $h_t$ = hidden state di posisi $t$
- $\alpha_t$ = **attention weight** (seberapa penting kata ke-$t$)
- $c$ = **context vector** (representasi teks yang sudah di-weight)

Keuntungan: kita bisa **visualisasikan** $\alpha_t$ untuk melihat kata mana yang paling berpengaruh!

```
BiLSTM hidden states: [h1, h2, h3, h4, h5]
                        ↓   ↓   ↓   ↓   ↓
Attention weights:   [0.1, 0.05, 0.15, 0.6, 0.1]  ← "anxious" = 0.6
                        ↓
Context vector = weighted sum
                        ↓
FC → 7 kelas
```

In [ ]:
from models import BiLSTMAttentionClassifier
from config import BILSTM_ATT_MODEL_PATH

# Inisialisasi model
bilstm_att = BiLSTMAttentionClassifier(vocab_size=len(vocab))
bilstm_att = bilstm_att.to(DEVICE)

print("Arsitektur BiLSTM+Attention:")
print(bilstm_att)
print(f"\n📊 Total parameter: {count_parameters(bilstm_att):,}")

In [ ]:
print("🚀 Mulai training BiLSTM+Attention...")
hist_bilstm_att = train_model(
    model=bilstm_att,
    train_loader=train_loader,
    val_loader=val_loader,
    model_type="lstm_att",
    save_path=BILSTM_ATT_MODEL_PATH,
    epochs=LSTM_EPOCHS,
    lr=LSTM_LR,
    patience=LSTM_PATIENCE,
    device=DEVICE,
    label_counts=label_counts,
)

In [ ]:
# Evaluasi
_, _, preds_att, labels_att = evaluate_lstm(bilstm_att, test_loader, criterion, DEVICE)

plot_confusion_matrix(labels_att, preds_att, "BiLSTM+Attention")
metrics_att = print_classification_report(labels_att, preds_att, "BiLSTM+Attention")
metrics_att["training_time_min"] = hist_bilstm_att["total_time"] / 60

plot_training_curves(hist_bilstm_att, "BiLSTM+Attention")

In [ ]:
from train import predict_single_lstm, plot_attention_heatmap
from preprocess import clean_text

# Visualisasi attention pada contoh teks
sample_texts = [
    "I feel so hopeless and empty inside, nothing brings me joy anymore",
    "I have been feeling very anxious and worried about everything lately",
    "Today was a great day, I feel happy and grateful for my life",
]

for text in sample_texts:
    result = predict_single_lstm(
        bilstm_att, text, vocab, DEVICE, return_attention=True
    )
    print(f"\nTeks: {text[:60]}...")
    print(f"Prediksi: {result['label']} (confidence: {result['confidence']:.3f})")
    
    if "attention_weights" in result:
        cleaned = clean_text(text)
        plot_attention_heatmap(
            cleaned, result["attention_weights"],
            model_name=f"BiLSTM+Attention → {result['label']}",
            save=False,
        )

### Interpretasi Attention

Dari heatmap attention di atas, perhatikan:

- **Warna lebih gelap/merah** = bobot attention **tinggi** → kata ini **lebih berpengaruh** pada prediksi model.
- **Warna lebih terang/kuning** = bobot attention **rendah** → model kurang memperhatikan kata ini.

Contoh yang diharapkan:
- Teks dengan label `Depression`: kata seperti **"hopeless"**, **"empty"**, **"nothing"** seharusnya mendapat attention tinggi.
- Teks dengan label `Anxiety`: kata **"anxious"**, **"worried"**, **"nervous"** mendapat perhatian lebih.

> 💡 Ini adalah salah satu kekuatan **Explainable AI (XAI)** — kita bisa menjelaskan *mengapa* model membuat prediksi tertentu.

---

## 🤗 Bagian 7: Model 3 — DistilBERT Fine-tuning

### Apa itu Transformer & BERT?

**Transformer** (Vaswani et al., 2017) adalah arsitektur berbasis **self-attention** yang memproses seluruh urutan sekaligus (bukan sekuensial seperti LSTM).

**BERT** (Devlin et al., 2018) adalah model Transformer yang di-*pre-train* pada:
1. **Masked Language Modeling** — prediksi kata yang disembunyikan
2. **Next Sentence Prediction** — prediksi apakah kalimat B mengikuti kalimat A

**DistilBERT** adalah versi BERT yang lebih kecil dan cepat (hasil distilasi pengetahuan):
- 40% lebih kecil dari BERT-base
- 60% lebih cepat
- 97% akurasi BERT pada benchmark NLP

### Pre-training vs Fine-tuning

```
Pre-training (sudah dilakukan, kita pakai hasilnya):
  Corpus besar (Wikipedia + BooksCorpus)
      → DistilBERT belajar representasi bahasa umum

Fine-tuning (kita lakukan sekarang):
  Dataset mental health spesifik
      → DistilBERT + classification head
      → Adaptasi untuk tugas klasifikasi 7 kelas
```

**Keuntungan Fine-tuning:**
- Tidak perlu data banyak karena model sudah "mengerti" bahasa
- Training lebih cepat (hanya beberapa epoch)
- Performa umumnya lebih baik dari model dari scratch

In [ ]:
from transformers import DistilBertTokenizerFast
from dataset import get_bert_dataloaders
from config import BERT_MODEL, BERT_MAX_LEN, BERT_BATCH_SIZE

# Load tokenizer DistilBERT
print("Memuat tokenizer DistilBERT...")
tokenizer = DistilBertTokenizerFast.from_pretrained(BERT_MODEL)

# Contoh tokenisasi
contoh = "I feel very anxious and depressed"
enc = tokenizer(contoh, return_tensors="pt")
print(f"\nContoh teks  : '{contoh}'")
print(f"Input IDs    : {enc['input_ids'][0].tolist()}")
print(f"Tokens       : {tokenizer.convert_ids_to_tokens(enc['input_ids'][0])}")

# Buat DataLoaders
train_bert, val_bert, test_bert = get_bert_dataloaders(
    df, tokenizer, max_len=BERT_MAX_LEN, batch_size=BERT_BATCH_SIZE
)

In [ ]:
from models import DistilBERTClassifier
from config import DISTILBERT_MODEL_DIR
import os

# Inisialisasi model
distilbert = DistilBERTClassifier()
distilbert = distilbert.to(DEVICE)

print("Arsitektur DistilBERT Classifier:")
print(distilbert)
print(f"\n📊 Total parameter: {count_parameters(distilbert):,}")

In [ ]:
from config import BERT_EPOCHS, BERT_LR, BERT_PATIENCE

os.makedirs(DISTILBERT_MODEL_DIR, exist_ok=True)
bert_save_path = os.path.join(DISTILBERT_MODEL_DIR, "distilbert.pt")

print("🚀 Mulai fine-tuning DistilBERT...")
print("   (Ini lebih lambat dari BiLSTM — harap bersabar ☕)")

hist_bert = train_model(
    model=distilbert,
    train_loader=train_bert,
    val_loader=val_bert,
    model_type="bert",
    save_path=bert_save_path,
    epochs=BERT_EPOCHS,
    lr=BERT_LR,
    patience=BERT_PATIENCE,
    device=DEVICE,
    label_counts=label_counts,
)

In [ ]:
from train import evaluate_bert

_, _, preds_bert, labels_bert = evaluate_bert(
    distilbert, test_bert, get_criterion(label_counts), DEVICE
)

plot_confusion_matrix(labels_bert, preds_bert, "DistilBERT")
metrics_bert = print_classification_report(labels_bert, preds_bert, "DistilBERT")
metrics_bert["training_time_min"] = hist_bert["total_time"] / 60

plot_training_curves(hist_bert, "DistilBERT")

---

## 📊 Bagian 8: Komparasi Model

Setelah melatih ketiga model, mari kita bandingkan performa mereka.

### Metrik yang Digunakan

| Metrik | Penjelasan |
|--------|------------|
| **Accuracy** | % prediksi benar dari total data |
| **F1 Macro** | Rata-rata F1 per kelas (semua kelas dianggap sama penting) |
| **F1 Weighted** | Rata-rata F1 per kelas dibobot jumlah sampel |

> Karena dataset tidak seimbang, **F1 Macro** adalah metrik utama yang lebih adil.

In [ ]:
from train import compare_models

all_results = {
    "BiLSTM":           metrics_bilstm,
    "BiLSTM+Attention": metrics_att,
    "DistilBERT":       metrics_bert,
}

# Bar chart perbandingan
compare_models(all_results)

In [ ]:
# Tabel perbandingan lengkap
import pandas as pd

compare_df = pd.DataFrame([
    {
        "Model":            name,
        "Accuracy":         f"{m['accuracy']:.4f}",
        "F1 Macro":         f"{m['f1_macro']:.4f}",
        "F1 Weighted":      f"{m['f1_weighted']:.4f}",
        "Waktu Training":   f"{m['training_time_min']:.1f} menit",
    }
    for name, m in all_results.items()
])

print("\n📊 TABEL PERBANDINGAN MODEL")
print("=" * 70)
print(compare_df.to_string(index=False))
compare_df

### Diskusi: Trade-off Kompleksitas vs Performa

| Aspek | BiLSTM | BiLSTM+Attention | DistilBERT |
|-------|--------|-----------------|------------|
| **Arsitektur** | Recurrent | Recurrent + Attention | Transformer |
| **Embedding** | Train from scratch | Train from scratch | Pre-trained |
| **Parameter** | ~6–7 juta | ~7 juta | ~67 juta |
| **Waktu training** | Cepat | Cepat | Lambat |
| **Performa** | Sedang | Lebih baik | Terbaik |
| **Interpretabilitas** | Terbatas | Attention heatmap | Terbatas |
| **Kebutuhan data** | Banyak | Banyak | Sedikit |
| **Kebutuhan GPU** | Rendah | Rendah | Tinggi |

**Pilih model berdasarkan kebutuhan:**
- 🔵 **BiLSTM** — Sumber daya terbatas, butuh solusi cepat
- 🟡 **BiLSTM+Attention** — Butuh interpretabilitas (XAI)
- 🟢 **DistilBERT** — Prioritas akurasi, ada GPU

---

## 🔍 Bagian 9: Inferensi Interaktif

Mari kita coba ketiga model pada teks baru!

In [ ]:
from train import predict_single_lstm, predict_single_bert

# Contoh teks untuk dicoba
test_sentences = [
    "I have been feeling hopeless and worthless for weeks. I don't see any reason to continue.",
    "Everything makes me nervous. I can't stop worrying about what might happen.",
    "I had a really productive day. Feeling energized and positive about the future.",
    "My mood swings are uncontrollable. One moment I'm on top of the world, next I'm in despair.",
    "I've been under a lot of pressure at work and can't seem to relax at all.",
]

print("\n" + "="*75)
print(" INFERENSI KETIGA MODEL")
print("="*75)

for text in test_sentences:
    r1 = predict_single_lstm(bilstm, text, vocab, DEVICE)
    r2 = predict_single_lstm(bilstm_att, text, vocab, DEVICE)
    r3 = predict_single_bert(distilbert, text, tokenizer, DEVICE)

    print(f"\n📝 Teks: {text[:70]}..." if len(text) > 70 else f"\n📝 Teks: {text}")
    print(f"   BiLSTM           : {r1['label']:<22} (conf: {r1['confidence']:.3f})")
    print(f"   BiLSTM+Attention : {r2['label']:<22} (conf: {r2['confidence']:.3f})")
    print(f"   DistilBERT       : {r3['label']:<22} (conf: {r3['confidence']:.3f})")

In [ ]:
# Coba teks Anda sendiri!
teks_anda = "I feel very sad and lonely today, nothing seems to go right"  # ganti dengan teks Anda

print(f"Teks input: '{teks_anda}'\n")

r_bilstm  = predict_single_lstm(bilstm,     teks_anda, vocab,     DEVICE)
r_att     = predict_single_lstm(bilstm_att, teks_anda, vocab,     DEVICE, return_attention=True)
r_bert    = predict_single_bert(distilbert, teks_anda, tokenizer, DEVICE)

print(f"{'Model':<20} {'Prediksi':<25} {'Confidence':>12}")
print("-" * 60)
print(f"{'BiLSTM':<20} {r_bilstm['label']:<25} {r_bilstm['confidence']:>12.4f}")
print(f"{'BiLSTM+Attention':<20} {r_att['label']:<25} {r_att['confidence']:>12.4f}")
print(f"{'DistilBERT':<20} {r_bert['label']:<25} {r_bert['confidence']:>12.4f}")

# Tampilkan attention heatmap
if "attention_weights" in r_att:
    from preprocess import clean_text
    cleaned = clean_text(teks_anda)
    print(f"\nAttention Heatmap (BiLSTM+Attention):")
    plot_attention_heatmap(
        cleaned, r_att["attention_weights"], 
        model_name=f"Prediksi: {r_att['label']}",
        save=False
    )

---

## 💾 Bagian 10: Simpan Model

Pastikan semua model tersimpan dengan benar sebelum melanjutkan ke deployment.

In [ ]:
import os
from config import BILSTM_MODEL_PATH, BILSTM_ATT_MODEL_PATH, DISTILBERT_MODEL_DIR, VOCAB_PATH

print("📦 Status penyimpanan model:")
files_to_check = [
    (BILSTM_MODEL_PATH,     "BiLSTM"),
    (BILSTM_ATT_MODEL_PATH, "BiLSTM+Attention"),
    (os.path.join(DISTILBERT_MODEL_DIR, "distilbert.pt"), "DistilBERT"),
    (VOCAB_PATH,            "Vocabulary"),
]

for path, name in files_to_check:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f"  ✅ {name:<22}: {path} ({size_mb:.1f} MB)")
    else:
        print(f"  ❌ {name:<22}: TIDAK DITEMUKAN — {path}")

---

## 🎓 Bagian 11: Ringkasan & Penutup

### Apa yang Sudah Kita Pelajari

✅ **Deep Learning untuk NLP** — tiga pendekatan berbeda untuk klasifikasi teks  
✅ **BiLSTM** — bagaimana LSTM bidirectional menangkap konteks dari dua arah  
✅ **Attention Mechanism** — cara model "memperhatikan" kata-kata penting  
✅ **Transfer Learning** — manfaat fine-tuning model pre-trained (DistilBERT)  
✅ **Handling class imbalance** — weighted CrossEntropyLoss  
✅ **Evaluasi komprehensif** — F1 Macro, Confusion Matrix, Classification Report  
✅ **Explainability** — attention heatmap untuk interpretasi model  

### Langkah Selanjutnya

1. **Deployment** — Deploy model ke Hugging Face Space (lihat folder `hf_space/`)
2. **Eksperimen** — Coba ubah `SAMPLE_SIZE=None` untuk train dengan data penuh
3. **Hyperparameter tuning** — Ubah `EMBED_DIM`, `HIDDEN_DIM`, `LSTM_LR` di `config.py`
4. **Augmentasi data** — Untuk kelas yang sedikit datanya (Bipolar, Personality Disorder)
5. **BERT lain** — Coba `bert-base-uncased` atau `mental/mental-bert-base-uncased`

---

### Referensi

1. Hochreiter, S., & Schmidhuber, J. (1997). Long short-term memory. *Neural Computation*, 9(8), 1735–1780.
2. Bahdanau, D., Cho, K., & Bengio, Y. (2015). Neural machine translation by jointly learning to align and translate. *ICLR 2015*.
3. Vaswani, A., et al. (2017). Attention is all you need. *NeurIPS 2017*.
4. Devlin, J., Chang, M. W., Lee, K., & Toutanova, K. (2019). BERT: Pre-training of deep bidirectional transformers for language understanding. *NAACL 2019*.
5. Sanh, V., et al. (2019). DistilBERT, a distilled version of BERT: smaller, faster, cheaper and lighter. *arXiv:1910.01108*.
6. Sarkar, S. (2023). Sentiment Analysis for Mental Health Dataset. *Kaggle*.

---
_Workshop PBA — Institut Teknologi Sumatera (ITERA), 2026_